# Week 6 Exercise — Product Review Sentiment Classifier

This notebook demonstrates how to fine-tune a language model to perform
sentiment classification on product reviews. A subset of labeled reviews
is taken from the Hugging Face Amazon Polarity dataset and formatted into
the JSONL structure required for OpenAI fine-tuning.

The fine-tuned model learns to classify reviews as either **positive** or
**negative** based on examples provided during training. After training
completes, the model can be used to predict the sentiment of new product
reviews using a simple inference function.

This exercise focuses on understanding the fine-tuning workflow:
1. Preparing labeled training data
2. Formatting data for OpenAI fine-tuning
3. Creating and monitoring a fine-tuning job
4. Using the resulting fine-tuned model for predictions


In [15]:
import os
import json
import random
from dotenv import load_dotenv
from datasets import load_dataset
from openai import OpenAI

load_dotenv()

client = OpenAI()

api_key = os.getenv("OPENAI_API_KEY")

if api_key:
    print("OpenAI API key loaded.")
else:
    print("OpenAI API key not found.")

OpenAI API key loaded.


In [16]:
dataset = load_dataset("imdb")

train_data = dataset["train"]
test_data = dataset["test"]

print("Train samples:", len(train_data))
print("Test samples:", len(test_data))

print("\nExample review:")
print(train_data[0]["text"])
print("\nLabel:", train_data[0]["label"])

KeyboardInterrupt: 

In [ ]:
def label_to_sentiment(label):
    return "positive" if label == 1 else "negative"

In [ ]:
def messages_for_training(example):

    return {
        "messages": [
            {"role": "system", "content": "Classify the sentiment of the product review. Reply only with 'positive' or 'negative'."},
            {"role": "user", "content": example["text"][:1000]},
            {"role": "assistant", "content": label_to_sentiment(example["label"])}
        ]
    }

In [ ]:
print(json.dumps(messages_for_training(train_data[0]), indent=2))

In [ ]:
TRAIN_SIZE = 50
VAL_SIZE = 20

train_subset = train_data.shuffle(seed=42).select(range(TRAIN_SIZE))
val_subset = train_data.shuffle(seed=42).select(range(TRAIN_SIZE, TRAIN_SIZE + VAL_SIZE))


def write_jsonl(data, filename):

    with open(filename, "w") as f:
        for example in data:
            json.dump(messages_for_training(example), f)
            f.write("\n")


write_jsonl(train_subset, "train.jsonl")
write_jsonl(val_subset, "val.jsonl")

print("Training and validation files created.")

In [ ]:
with open("train.jsonl", "rb") as f:
    train_file = client.files.create(file=f, purpose="fine-tune")

with open("val.jsonl", "rb") as f:
    val_file = client.files.create(file=f, purpose="fine-tune")

print("Train file ID:", train_file.id)
print("Validation file ID:", val_file.id)

In [ ]:
job = client.fine_tuning.jobs.create(
    training_file=train_file.id,
    validation_file=val_file.id,
    model="gpt-4o-mini-2024-07-18",
    hyperparameters={"n_epochs": 1},
    suffix="review-sentiment"
)

print("Fine-tuning job created.")
print("Job ID:", job.id)

In [ ]:
job_status = client.fine_tuning.jobs.retrieve(job.id)

print("Status:", job_status.status)

if job_status.fine_tuned_model:
    MODEL_NAME = job_status.fine_tuned_model
    print("Model ready:", MODEL_NAME)

    if job_status.trained_tokens:
        print("Trained tokens:", job_status.trained_tokens)
else:
    print("Training still in progress. Run this cell again in a few minutes.")

In [ ]:
MODEL_NAME = job_status.fine_tuned_model

def predict_sentiment(text):

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": "Classify the sentiment of the product review. Reply only with 'positive' or 'negative'."},
            {"role": "user", "content": text}
        ],
        max_tokens=5
    )

    return response.choices[0].message.content.strip()

In [ ]:
example = test_data[0]

print("Review:")
print(example["text"])

print("\nActual label:", label_to_sentiment(example["label"]))

print("\nPrediction:")
print(predict_sentiment(example["text"]))

In [ ]:
TEST_SIZE = 50

correct = 0

for i in range(TEST_SIZE):

    item = test_data[i]

    prediction = predict_sentiment(item["text"])
    actual = label_to_sentiment(item["label"])

    if prediction == actual:
        correct += 1

accuracy = correct / TEST_SIZE

print("Accuracy:", accuracy)

In [ ]:
import gradio as gr

def classify_review(review):
    return predict_sentiment(review)

demo = gr.Interface(
    fn=classify_review,
    inputs=gr.Textbox(label="Product Review"),
    outputs=gr.Textbox(label="Sentiment"),
    title="Product Review Sentiment Classifier"
)

demo.launch()